## Setup and Imports

In [1]:
import sys
import time
import tracemalloc
import os
from pathlib import Path
from typing import List, Dict, Any
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add backend to path - use absolute path from notebook location
notebook_dir = Path(os.path.abspath('')).resolve()
if notebook_dir.name == 'notebooks':
    backend_path = notebook_dir.parent / 'backend'
else:
    # Fallback: assume we're already in the repo root
    backend_path = notebook_dir / 'backend'

sys.path.insert(0, str(backend_path))

# Import parsers
from app.rag.parsers import ParserFactory, get_parser_factory
from app.rag.parsers.code_parser import CodeParser
from app.rag.parsers.text_parser import TextParser
from app.rag.parsers.base_parser import ParsedContent

# Import chunkers
from app.rag.chunkers import ChunkerFactory, get_chunker_factory
from app.rag.chunkers.text_chunker import TextChunker
from app.rag.chunkers.code_chunker import CodeChunker
from app.rag.chunkers.base_chunker import Chunk

# Config
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Imports successful")

/home/adam/repos/AI-Assistant-for-Technical-Knowledge-Management/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Imports successful


## Test Data Preparation

In [2]:
# Create temporary test files
import tempfile
import os

test_dir = Path(tempfile.mkdtemp())
print(f"Test directory: {test_dir}")

# Sample Python code
python_code = '''"""Sample Python module for testing."""

def calculate_distance(x1, y1, x2, y2):
    """Calculate Euclidean distance between two points."""
    dx = x2 - x1
    dy = y2 - y1
    return (dx**2 + dy**2)**0.5

class Point:
    """Represents a 2D point."""
    
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def distance_to(self, other):
        """Calculate distance to another point."""
        return calculate_distance(self.x, self.y, other.x, other.y)
'''

# Sample C++ code
cpp_code = '''// Geometry utilities
#include <cmath>

// Calculate distance between two points
double calculate_distance(double x1, double y1, double x2, double y2) {
    double dx = x2 - x1;
    double dy = y2 - y1;
    return sqrt(dx*dx + dy*dy);
}

class Point {
public:
    double x, y;
    
    Point(double x, double y) : x(x), y(y) {}
    
    double distanceTo(const Point& other) const {
        return calculate_distance(x, y, other.x, other.y);
    }
};
'''

# Sample text document
text_content = '''Introduction to RAG Systems

Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with language generation. 
It works by first retrieving relevant documents from a knowledge base, then using those documents as context for 
generating responses.

Key Components:
1. Document parsing and chunking
2. Embedding generation
3. Vector storage and retrieval
4. Context-aware generation

This approach significantly improves the accuracy and relevance of generated responses by grounding them in 
actual documented knowledge rather than relying solely on the model's training data.
'''

# Write test files
test_files = {
    'test.py': python_code,
    'test.cpp': cpp_code,
    'document.txt': text_content
}

for filename, content in test_files.items():
    (test_dir / filename).write_text(content)

print(f"✓ Created {len(test_files)} test files")
for filename in test_files:
    print(f"  - {filename}")

Test directory: /tmp/tmp5e7wpryp
✓ Created 3 test files
  - test.py
  - test.cpp
  - document.txt


## 1. Parser Unit Tests

### 1.1 ParserFactory Tests

In [3]:
def test_parser_factory():
    """Test ParserFactory initialization and parser selection."""
    results = []
    
    factory = get_parser_factory()
    
    test_cases = [
        ('test.py', CodeParser, True),
        ('test.cpp', CodeParser, True),
        ('document.txt', TextParser, True),
        ('unknown.xyz', None, False),
    ]
    
    for filename, expected_parser_class, should_support in test_cases:
        file_path = test_dir / filename
        parser = factory.get_parser(file_path)
        
        # Test support
        supports = factory.supports(file_path)
        
        # Validate
        if should_support:
            test_passed = (
                parser is not None and 
                isinstance(parser, expected_parser_class) and
                supports
            )
        else:
            test_passed = parser is None and not supports
        
        results.append({
            'Test': f'ParserFactory: {filename}',
            'Expected': expected_parser_class.__name__ if expected_parser_class else 'None',
            'Got': parser.__class__.__name__ if parser else 'None',
            'Supports': supports,
            'Pass': '✓' if test_passed else '✗'
        })
    
    # Test cache
    parser1 = factory.get_parser(test_dir / 'test.py')
    parser2 = factory.get_parser(test_dir / 'test.py')
    cache_works = parser1 is parser2  # Same instance
    
    results.append({
        'Test': 'ParserFactory: Cache',
        'Expected': 'Same instance',
        'Got': 'Same instance' if cache_works else 'Different instances',
        'Supports': True,
        'Pass': '✓' if cache_works else '✗'
    })
    
    return pd.DataFrame(results)

parser_factory_results = test_parser_factory()
print("\n=== ParserFactory Tests ===")
print(parser_factory_results.to_string(index=False))
print(f"\nPassed: {(parser_factory_results['Pass'] == '✓').sum()}/{len(parser_factory_results)}")

2025-12-17 16:15:23,303 - INFO - ✓ Docling DocumentConverter initialized
2025-12-17 16:15:23,309 - INFO - ✓ Docling DocumentConverter initialized for DOCX
2025-12-17 16:15:23,314 - INFO - ✓ Docling DocumentConverter initialized for PPTX
2025-12-17 16:15:23,315 - INFO - ✓ Initialized Tree-sitter parsers for: ['cpp', 'python', 'java', 'javascript']
2025-12-17 16:15:23,316 - WARNING - ImageParser is a stub. Vision-based parsing not yet implemented.
2025-12-17 16:15:23,317 - INFO - ✓ ParserFactory initialized with 6 parsers
2025-12-17 16:15:23,318 - WARNING - No parser found for file: /tmp/tmp5e7wpryp/unknown.xyz
2025-12-17 16:15:23,319 - WARNING - No parser found for file: /tmp/tmp5e7wpryp/unknown.xyz



=== ParserFactory Tests ===
                       Test      Expected           Got  Supports Pass
     ParserFactory: test.py    CodeParser    CodeParser      True    ✓
    ParserFactory: test.cpp    CodeParser    CodeParser      True    ✓
ParserFactory: document.txt    TextParser    TextParser      True    ✓
 ParserFactory: unknown.xyz          None          None     False    ✓
       ParserFactory: Cache Same instance Same instance      True    ✓

Passed: 5/5


### 1.2 CodeParser Tests

In [4]:
def test_code_parser():
    """Test CodeParser with Python and C++ files."""
    results = []
    parser = CodeParser()
    
    # Test Python
    py_file = test_dir / 'test.py'
    parsed_py = parser.parse(py_file)
    
    py_units = parsed_py.metadata.get('units', [])
    py_has_function = any(u.get('type') == 'function' and u.get('name') == 'calculate_distance' for u in py_units)
    py_has_class = any(u.get('type') == 'class' and u.get('name') == 'Point' for u in py_units)
    
    results.append({
        'Test': 'CodeParser: Python functions',
        'Expected': 'calculate_distance found',
        'Got': f'{len(py_units)} units, function found' if py_has_function else 'Not found',
        'Pass': '✓' if py_has_function else '✗'
    })
    
    results.append({
        'Test': 'CodeParser: Python classes',
        'Expected': 'Point class found',
        'Got': f'{len(py_units)} units, class found' if py_has_class else 'Not found',
        'Pass': '✓' if py_has_class else '✗'
    })
    
    # Test C++
    cpp_file = test_dir / 'test.cpp'
    parsed_cpp = parser.parse(cpp_file)
    
    cpp_units = parsed_cpp.metadata.get('units', [])
    cpp_has_functions = any(u.get('type') == 'function' for u in cpp_units)
    cpp_has_correct_content = 'calculate_distance' in parsed_cpp.text
    
    results.append({
        'Test': 'CodeParser: C++ functions',
        'Expected': 'Functions detected + content OK',
        'Got': f'{len(cpp_units)} units, functions found' if (cpp_has_functions and cpp_has_correct_content) else 'Not found',
        'Pass': '✓' if (cpp_has_functions and cpp_has_correct_content) else '✗'
    })
    
    # Test content type
    results.append({
        'Test': 'CodeParser: content_type',
        'Expected': 'code',
        'Got': parsed_py.content_type,
        'Pass': '✓' if parsed_py.content_type == 'code' else '✗'
    })
    
    return pd.DataFrame(results)

code_parser_results = test_code_parser()
print("\n=== CodeParser Tests ===")
print(code_parser_results.to_string(index=False))
print(f"\nPassed: {(code_parser_results['Pass'] == '✓').sum()}/{len(code_parser_results)}")

2025-12-17 16:15:33,620 - INFO - ✓ Initialized Tree-sitter parsers for: ['cpp', 'python', 'java', 'javascript']
2025-12-17 16:15:33,622 - INFO - Parsing code file: test.py (python)
2025-12-17 16:15:33,626 - INFO - ✓ Parsed test.py: 476 chars, 4 semantic units
2025-12-17 16:15:33,628 - INFO - Parsing code file: test.cpp (cpp)
2025-12-17 16:15:33,640 - INFO - ✓ Parsed test.cpp: 443 chars, 4 semantic units



=== CodeParser Tests ===
                        Test                        Expected                      Got Pass
CodeParser: Python functions        calculate_distance found  4 units, function found    ✓
  CodeParser: Python classes               Point class found     4 units, class found    ✓
   CodeParser: C++ functions Functions detected + content OK 4 units, functions found    ✓
    CodeParser: content_type                            code                     code    ✓

Passed: 4/4


### 1.3 TextParser Tests

In [5]:
def test_text_parser():
    """Test TextParser with text file."""
    results = []
    parser = TextParser()
    
    text_file = test_dir / 'document.txt'
    parsed = parser.parse(text_file)
    
    # Check content extraction
    has_content = 'RAG' in parsed.text and 'Retrieval-Augmented Generation' in parsed.text
    results.append({
        'Test': 'TextParser: content extraction',
        'Expected': 'Contains RAG content',
        'Got': 'Found' if has_content else 'Not found',
        'Pass': '✓' if has_content else '✗'
    })
    
    # Check content type
    results.append({
        'Test': 'TextParser: content_type',
        'Expected': 'text',
        'Got': parsed.content_type,
        'Pass': '✓' if parsed.content_type == 'text' else '✗'
    })
    
    # Check text length
    results.append({
        'Test': 'TextParser: text length',
        'Expected': '> 100 chars',
        'Got': f'{len(parsed.text)} chars',
        'Pass': '✓' if len(parsed.text) > 100 else '✗'
    })
    
    return pd.DataFrame(results)

text_parser_results = test_text_parser()
print("\n=== TextParser Tests ===")
print(text_parser_results.to_string(index=False))
print(f"\nPassed: {(text_parser_results['Pass'] == '✓').sum()}/{len(text_parser_results)}")

2025-12-17 16:17:52,101 - INFO - Parsing text file: document.txt



=== TextParser Tests ===
                          Test             Expected       Got Pass
TextParser: content extraction Contains RAG content     Found    ✓
      TextParser: content_type                 text      text    ✓
       TextParser: text length          > 100 chars 609 chars    ✓

Passed: 3/3


## 2. Chunker Unit Tests

### 2.1 ChunkerFactory Tests

In [6]:
def test_chunker_factory():
    """Test ChunkerFactory selection logic."""
    results = []
    factory = get_chunker_factory()
    
    # Test code content
    code_content = ParsedContent(
        text="def test(): pass",
        content_type="code",
        language="python",
        file_path=Path("test.py"),
        metadata={}
    )
    
    code_chunker = factory.get_chunker(code_content)
    results.append({
        'Test': 'ChunkerFactory: code content',
        'Expected': 'CodeChunker',
        'Got': code_chunker.__class__.__name__,
        'Pass': '✓' if isinstance(code_chunker, CodeChunker) else '✗'
    })
    
    # Test text content
    text_content = ParsedContent(
        text="This is some text.",
        content_type="text",
        language="en",
        file_path=Path("test.txt"),
        metadata={}
    )
    
    text_chunker = factory.get_chunker(text_content)
    results.append({
        'Test': 'ChunkerFactory: text content',
        'Expected': 'TextChunker',
        'Got': text_chunker.__class__.__name__,
        'Pass': '✓' if isinstance(text_chunker, TextChunker) else '✗'
    })
    
    return pd.DataFrame(results)

chunker_factory_results = test_chunker_factory()
print("\n=== ChunkerFactory Tests ===")
print(chunker_factory_results.to_string(index=False))
print(f"\nPassed: {(chunker_factory_results['Pass'] == '✓').sum()}/{len(chunker_factory_results)}")

2025-12-17 16:18:18,309 - INFO - ✓ ChunkerFactory initialized



=== ChunkerFactory Tests ===
                        Test    Expected         Got Pass
ChunkerFactory: code content CodeChunker CodeChunker    ✓
ChunkerFactory: text content TextChunker TextChunker    ✓

Passed: 2/2


### 2.2 TextChunker Tests

In [7]:
def test_text_chunker():
    """Test TextChunker with various text sizes."""
    results = []
    chunker = TextChunker(chunk_size=100, chunk_overlap=20)
    
    # Test small text (no chunking needed)
    small_text = "This is a short text."
    small_content = ParsedContent(
        text=small_text,
        content_type="text",
        language="en",
        file_path=Path("small.txt"),
        metadata={}
    )
    
    small_chunks = chunker.chunk(small_content)
    results.append({
        'Test': 'TextChunker: small text',
        'Expected': '1 chunk',
        'Got': f'{len(small_chunks)} chunks',
        'Pass': '✓' if len(small_chunks) == 1 else '✗'
    })
    
    # Test large text (multiple chunks)
    large_text = (test_dir / 'document.txt').read_text()
    large_content = ParsedContent(
        text=large_text,
        content_type="text",
        language="en",
        file_path=Path("large.txt"),
        metadata={}
    )
    
    large_chunks = chunker.chunk(large_content)
    results.append({
        'Test': 'TextChunker: large text',
        'Expected': '> 1 chunk',
        'Got': f'{len(large_chunks)} chunks',
        'Pass': '✓' if len(large_chunks) > 1 else '✗'
    })
    
    # Test chunk size constraint
    max_size = max(len(c.text) for c in large_chunks)
    results.append({
        'Test': 'TextChunker: max chunk size',
        'Expected': '<= 150 chars (with tolerance)',
        'Got': f'{max_size} chars',
        'Pass': '✓' if max_size <= 150 else '✗'
    })
    
    # Test metadata
    has_metadata = all(
        'chunk_index' in c.metadata and 
        'total_chunks' in c.metadata
        for c in large_chunks
    )
    
    results.append({
        'Test': 'TextChunker: metadata completeness',
        'Expected': 'All chunks have chunk_index, total_chunks',
        'Got': 'Complete' if has_metadata else 'Incomplete',
        'Pass': '✓' if has_metadata else '✗'
    })
    
    return pd.DataFrame(results)

text_chunker_results = test_text_chunker()
print("\n=== TextChunker Tests ===")
print(text_chunker_results.to_string(index=False))
print(f"\nPassed: {(text_chunker_results['Pass'] == '✓').sum()}/{len(text_chunker_results)}")

2025-12-17 16:19:34,463 - INFO - Split small.txt into 1 chunks (original length: 21 chars)
2025-12-17 16:19:34,466 - INFO - Split large.txt into 12 chunks (original length: 608 chars)



=== TextChunker Tests ===
                              Test                                  Expected       Got Pass
           TextChunker: small text                                   1 chunk  1 chunks    ✓
           TextChunker: large text                                 > 1 chunk 12 chunks    ✓
       TextChunker: max chunk size             <= 150 chars (with tolerance)  99 chars    ✓
TextChunker: metadata completeness All chunks have chunk_index, total_chunks  Complete    ✓

Passed: 4/4


### 2.3 CodeChunker Tests

In [8]:
def test_code_chunker():
    """Test CodeChunker with parsed code."""
    results = []
    
    # Parse Python code first
    code_parser = CodeParser()
    py_file = test_dir / 'test.py'
    parsed_code = code_parser.parse(py_file)
    
    # Chunk it
    code_chunker = CodeChunker(chunk_size=800)
    chunks = code_chunker.chunk(parsed_code)
    
    # Test chunk count
    results.append({
        'Test': 'CodeChunker: chunk count',
        'Expected': '>= 2 chunks',
        'Got': f'{len(chunks)} chunks',
        'Pass': '✓' if len(chunks) >= 2 else '✗'
    })
    
    # Test unit metadata
    has_unit_metadata = all(
        'unit_type' in c.metadata and
        'unit_name' in c.metadata
        for c in chunks
    )
    
    results.append({
        'Test': 'CodeChunker: unit metadata',
        'Expected': 'All chunks have unit_type, unit_name',
        'Got': 'Complete' if has_unit_metadata else 'Incomplete',
        'Pass': '✓' if has_unit_metadata else '✗'
    })
    
    # Test complete functions
    function_chunk = next((c for c in chunks if c.metadata.get('unit_name') == 'calculate_distance'), None)
    if function_chunk:
        has_complete_function = (
            'def calculate_distance' in function_chunk.text and
            'return' in function_chunk.text
        )
        
        results.append({
            'Test': 'CodeChunker: complete function',
            'Expected': 'Function has def and return',
            'Got': 'Complete' if has_complete_function else 'Incomplete',
            'Pass': '✓' if has_complete_function else '✗'
        })
    
    return pd.DataFrame(results)

code_chunker_results = test_code_chunker()
print("\n=== CodeChunker Tests ===")
print(code_chunker_results.to_string(index=False))
print(f"\nPassed: {(code_chunker_results['Pass'] == '✓').sum()}/{len(code_chunker_results)}")

2025-12-17 16:20:05,253 - INFO - ✓ Initialized Tree-sitter parsers for: ['cpp', 'python', 'java', 'javascript']
2025-12-17 16:20:05,254 - INFO - Parsing code file: test.py (python)
2025-12-17 16:20:05,255 - INFO - ✓ Parsed test.py: 476 chars, 4 semantic units
2025-12-17 16:20:05,256 - INFO - Chunking test.py by 4 semantic units
2025-12-17 16:20:05,256 - INFO - ✓ Created 4 chunks from 4 units in test.py



=== CodeChunker Tests ===
                          Test                             Expected      Got Pass
      CodeChunker: chunk count                          >= 2 chunks 4 chunks    ✓
    CodeChunker: unit metadata All chunks have unit_type, unit_name Complete    ✓
CodeChunker: complete function          Function has def and return Complete    ✓

Passed: 3/3


## 3. Performance Benchmarks

### 3.1 Parser Performance

In [9]:
def benchmark_parsers():
    """Benchmark parsing speed and memory usage."""
    factory = get_parser_factory()
    benchmarks = []
    
    test_cases = [
        ('test.py', 'Python'),
        ('test.cpp', 'C++'),
        ('document.txt', 'Text'),
    ]
    
    for filename, file_type in test_cases:
        file_path = test_dir / filename
        parser = factory.get_parser(file_path)
        
        if parser is None:
            continue
        
        # Measure time
        start_time = time.time()
        
        # Measure memory
        tracemalloc.start()
        
        parsed = parser.parse(file_path)
        
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        
        elapsed = time.time() - start_time
        
        benchmarks.append({
            'File Type': file_type,
            'Parser': parser.__class__.__name__,
            'File Size (bytes)': file_path.stat().st_size,
            'Parse Time (ms)': f"{elapsed * 1000:.2f}",
            'Memory Peak (KB)': f"{peak / 1024:.2f}",
            'Text Length': len(parsed.text),
            'Metadata Keys': len(parsed.metadata)
        })
    
    return pd.DataFrame(benchmarks)

parser_benchmarks = benchmark_parsers()
print("\n=== Parser Performance Benchmarks ===")
print(parser_benchmarks.to_string(index=False))

2025-12-17 16:20:48,499 - INFO - Parsing code file: test.py (python)
2025-12-17 16:20:48,505 - INFO - ✓ Parsed test.py: 476 chars, 4 semantic units
2025-12-17 16:20:48,509 - INFO - Parsing code file: test.cpp (cpp)
2025-12-17 16:20:48,513 - INFO - ✓ Parsed test.cpp: 443 chars, 4 semantic units
2025-12-17 16:20:48,515 - INFO - Parsing text file: document.txt



=== Parser Performance Benchmarks ===
File Type     Parser  File Size (bytes) Parse Time (ms) Memory Peak (KB)  Text Length  Metadata Keys
   Python CodeParser                476            9.64            47.03          476              9
      C++ CodeParser                443            6.77            39.60          443              9
     Text TextParser                609            2.24             8.96          609              5


### 3.2 Chunker Performance

In [10]:
def benchmark_chunkers():
    """Benchmark chunking speed and memory usage."""
    benchmarks = []
    
    # Parse files first
    code_parser = CodeParser()
    py_parsed = code_parser.parse(test_dir / 'test.py')
    
    # Large text
    large_text = (test_dir / 'document.txt').read_text() * 10
    text_parsed = ParsedContent(
        text=large_text,
        content_type="text",
        language="en",
        file_path=Path("large.txt"),
        metadata={}
    )
    
    test_cases = [
        (py_parsed, 'Code', CodeChunker()),
        (text_parsed, 'Text', TextChunker()),
    ]
    
    for content, content_type, chunker in test_cases:
        # Measure time
        start_time = time.time()
        
        # Measure memory
        tracemalloc.start()
        
        chunks = chunker.chunk(content)
        
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        
        elapsed = time.time() - start_time
        
        # Calculate statistics
        chunk_sizes = [len(c.text) for c in chunks]
        
        benchmarks.append({
            'Content Type': content_type,
            'Chunker': chunker.__class__.__name__,
            'Input Size (chars)': len(content.text),
            'Chunk Time (ms)': f"{elapsed * 1000:.2f}",
            'Memory Peak (KB)': f"{peak / 1024:.2f}",
            'Chunk Count': len(chunks),
            'Avg Chunk Size': f"{sum(chunk_sizes) / len(chunk_sizes):.0f}",
            'Min/Max Size': f"{min(chunk_sizes)}/{max(chunk_sizes)}"
        })
    
    return pd.DataFrame(benchmarks)

chunker_benchmarks = benchmark_chunkers()
print("\n=== Chunker Performance Benchmarks ===")
print(chunker_benchmarks.to_string(index=False))

2025-12-17 16:22:08,468 - INFO - ✓ Initialized Tree-sitter parsers for: ['cpp', 'python', 'java', 'javascript']
2025-12-17 16:22:08,470 - INFO - Parsing code file: test.py (python)
2025-12-17 16:22:08,471 - INFO - ✓ Parsed test.py: 476 chars, 4 semantic units
2025-12-17 16:22:08,472 - INFO - Chunking test.py by 4 semantic units
2025-12-17 16:22:08,473 - INFO - ✓ Created 4 chunks from 4 units in test.py
2025-12-17 16:22:08,476 - INFO - Split large.txt into 7 chunks (original length: 6089 chars)



=== Chunker Performance Benchmarks ===
Content Type     Chunker  Input Size (chars) Chunk Time (ms) Memory Peak (KB)  Chunk Count Avg Chunk Size Min/Max Size
        Code CodeChunker                 476            3.00            12.89            4            161       63/268
        Text TextChunker                6090            1.23            22.65            7            868      579/993


## 4. Summary Report

In [ ]:
# Compile all test results
all_results = pd.concat([
    parser_factory_results,
    code_parser_results,
    text_parser_results,
    chunker_factory_results,
    text_chunker_results,
    code_chunker_results
], ignore_index=True)

total_tests = len(all_results)
passed_tests = (all_results['Pass'] == '✓').sum()
failed_tests = total_tests - passed_tests
pass_rate = (passed_tests / total_tests) * 100

print("\n" + "="*80)
print("FINAL TEST SUMMARY")
print("="*80)
print(f"\nTotal Tests: {total_tests}")
print(f"Passed: {passed_tests} ✓")
print(f"Failed: {failed_tests} ✗")
print(f"Pass Rate: {pass_rate:.1f}%")

if failed_tests > 0:
    print("\n Failed Tests:")
    print(all_results[all_results['Pass'] == '✗'][['Test', 'Expected', 'Got']].to_string(index=False))
else:
    print("\n All tests passed!")

print("\n" + "="*80)
print("PERFORMANCE SUMMARY")
print("="*80)
print("\nParser Performance:")
print(parser_benchmarks[['File Type', 'Parser', 'Parse Time (ms)', 'Memory Peak (KB)']].to_string(index=False))
print("\nChunker Performance:")
print(chunker_benchmarks[['Content Type', 'Chunker', 'Chunk Time (ms)', 'Chunk Count']].to_string(index=False))

print("\n" + "="*80)
print("✓ Benchmark suite completed")
print("="*80)


FINAL TEST SUMMARY

Total Tests: 21
Passed: 21 ✓
Failed: 0 ✗
Pass Rate: 100.0%

✅ All tests passed!

PERFORMANCE SUMMARY

Parser Performance:
File Type     Parser Parse Time (ms) Memory Peak (KB)
   Python CodeParser            9.64            47.03
      C++ CodeParser            6.77            39.60
     Text TextParser            2.24             8.96

Chunker Performance:
Content Type     Chunker Chunk Time (ms)  Chunk Count
        Code CodeChunker            3.00            4
        Text TextChunker            1.23            7

✓ Benchmark suite completed


## Cleanup

In [12]:
# Clean up temporary test files
import shutil

shutil.rmtree(test_dir)
print(f"✓ Cleaned up test directory: {test_dir}")

✓ Cleaned up test directory: /tmp/tmp5e7wpryp
